## Part 2: Simple Text Processing - Tokenization, Lemmatization, Word Frequency, Vectorization (20 pts)

Now we will start working on simple text processing using the `SpaCy` package and the same dataset as Part 1. The package should already be included in the `environment.yml`. However, we will also need to download `en_core_web_sm`, an English language text processing model. To do this, while having your `sotu` environment activated, run the following:

```
python -m spacy download en_core_web_sm
```

Now, you should be good to go!

Some important definitions:

- *Token*: a single word or piece of a word
- *Lemma*: the core component of a word, e.g., "complete" is the lemma for "completed" and "completely"
- *Stop Word*: a common word that does not add semantic value, such as "a", "and", "the", etc.
- *Vectorization*: representing a document as a vector where each index in the vector corresponds to a token or word and each entry is the count.

In this section, we will explore the most common tokens and lemmas throughout different slices of the speech data. We will also develop vectorization representations of the speeches. 

 The core steps are:

1. Process speeches using the SpaCy nlp module
2. Analyze Tokens vs Lemmas:
- Create a list of all tokens across all speeches that are not stop words, punctuation, or spaces.
- Create a second list of the lemmas for these same tokens.
- Display the top 25 for each of these and compare.
3. Analyze common word distributions over different years:
- Create a function that takes the dataset and a year as an input and outputs the top n lemmas for that year's speeches
- Compare the top 10 words for 2023 versus 2019
4. Document Vectorization:
- Train a Term Frequency-Inverse Document Frequency (TF-IDF) vectorization model using your processed dataset and scikit learn
- Output the feature vectors 

**Helpful Resources:**
- https://realpython.com/natural-language-processing-spacy-python/
- https://www.statology.org/text-preprocessing-feature-engineering-spacy/ 
- https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html# 
- https://www.geeksforgeeks.org/nlp/how-to-store-a-tfidfvectorizer-for-future-use-in-scikit-learn/ 


### Step 1: Process speeches using the SpaCy nlp module

In [25]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
from tqdm import tqdm
from collections import Counter

nlp = spacy.load("en_core_web_sm")

In [26]:
sou = pd.read_csv("data/SOTU.csv")
sou_21cen = sou[sou["Year"] >= 2000].reset_index(drop=True)
sou_21cen

,President,Year,Text,Word Count
0,Joseph R. Biden,2024.0,"\n[Before speaking, the President presented hi...",8003
1,Joseph R. Biden,2023.0,\nThe President. Mr. Speaker——\n[At this point...,8978
2,Joseph R. Biden,2022.0,"\nThe President. Thank you all very, very much...",7539
3,Joseph R. Biden,2021.0,\nThe President. Thank you. Thank you. Thank y...,7734
4,Donald J. Trump,2020.0,\nThe President. Thank you very much. Thank yo...,6169
5,Donald J. Trump,2019.0,"\nThe President. Madam Speaker, Mr. Vice Presi...",5519
6,Donald J. Trump,2018.0,"\nThe President. Mr. Speaker, Mr. Vice Preside...",5755
7,Donald J. Trump,2017.0,"\nThank you very much. Mr. Speaker, Mr. Vice P...",4903
8,Barack Obama,2016.0,"\nThank you. Mr. Speaker, Mr. Vice President, ...",5956
9,Barack Obama,2015.0,"\nThe President. Mr. Speaker, Mr. Vice Preside...",6659


### Step 2: Analyze Tokens vs Lemmas

In [27]:
processed_speech = []

for text in tqdm(sou_21cen["Text"], desc="Processing"):
    doc = nlp(text)
    processed_speech.append(doc)

Processing: 100%|██████████| 25/25 [00:23<00:00,  1.05it/s]


In [28]:
tokens = []

for doc in processed_speech:
    for token in doc:
        if not token.is_stop and not token.is_punct and not token.is_space:
            tokens.append(token.text.lower())

token_counts = Counter(tokens)
top_tokens = token_counts.most_common(25)
top_tokens

[('america', 816),
 ('people', 637),
 ('american', 582),
 ('new', 530),
 ('years', 439),
 ('americans', 437),
 ('world', 425),
 ('year', 406),
 ('country', 369),
 ('jobs', 348),
 ('tonight', 344),
 ('work', 324),
 ('know', 323),
 ('let', 320),
 ('congress', 317),
 ('nation', 311),
 ('time', 301),
 ('help', 282),
 ('need', 266),
 ('tax', 255),
 ('president', 247),
 ('economy', 243),
 ('like', 241),
 ('right', 240),
 ('want', 237)]

In [29]:
lemmas = []

for doc in processed_speech:
    for token in doc:
        if not token.is_stop and not token.is_punct and not token.is_space:
            lemmas.append(token.lemma_.lower())

lemma_counts = Counter(lemmas)
top_lemmas = lemma_counts.most_common(25)
top_lemmas

[('year', 845),
 ('america', 816),
 ('people', 639),
 ('american', 587),
 ('work', 557),
 ('new', 532),
 ('job', 486),
 ('country', 435),
 ('americans', 432),
 ('world', 426),
 ('know', 395),
 ('nation', 388),
 ('help', 378),
 ('need', 353),
 ('time', 351),
 ('tonight', 344),
 ('child', 332),
 ('let', 326),
 ('congress', 317),
 ('come', 301),
 ('family', 296),
 ('good', 294),
 ('right', 282),
 ('million', 274),
 ('want', 264)]

In [30]:
compare_df = pd.DataFrame({
    "Token": [w for w, _ in top_tokens],
    "Token Count": [c for _, c in top_tokens],
    "Lemma": [w for w, _ in top_lemmas],
    "Lemma Count": [c for _, c in top_lemmas]
})

compare_df

,Token,Token Count,Lemma,Lemma Count
0,america,816,year,845
1,people,637,america,816
2,american,582,people,639
3,new,530,american,587
4,years,439,work,557
5,americans,437,new,532
6,world,425,job,486
7,year,406,country,435
8,country,369,americans,432
9,jobs,348,world,426


In the token list, words like “year” and “years” appear separately, so their counts are split across two entries (439 for years and 406 for year). After lemmatization, these are merged into the single lemma “year”, which now shows a higher combined count of 845, giving a clearer picture of how often this concept appears. A similar pattern shows up for “job/jobs”. In the token table, jobs appears with a count of 348, but once we normalize to the lemma “job”, the total frequency rises to 486. These examples illustrates how lemmas group different grammatical forms together, revealing the underlying themes more clearly than raw tokens.

### Step 3: Analyze common word distributions over different years:

In [31]:
def top_lemmas_by_year(df, year, n=10):
    docs = nlp.pipe(df[df["Year"] == year]["Text"])
    lemmas = [
        token.lemma_.lower()
        for doc in docs
        for token in doc
        if not token.is_stop and not token.is_punct and not token.is_space
    ]
    return Counter(lemmas).most_common(n)

In [32]:
top_2023 = top_lemmas_by_year(sou_21cen, 2023, 10)
top_2019 = top_lemmas_by_year(sou_21cen, 2019, 10)

In [33]:
compare_2019_2023_df = pd.DataFrame({
    "2023 Lemma": [w for w, _ in top_2023],
    "2023 Count": [c for _, c in top_2023],
    "2019 Lemma": [w for w, _ in top_2019],
    "2019 Count": [c for _, c in top_2019]
})

compare_2019_2023_df

,2023 Lemma,2023 Count,2019 Lemma,2019 Count
0,year,58,year,38
1,go,56,american,34
2,let,45,thank,29
3,know,40,america,25
4,people,39,new,22
5,job,38,states,19
6,america,36,tonight,19
7,come,33,country,19
8,law,33,americans,18
9,pay,33,united,17


We see 2023 uses more economy-focused language (e.g., job, pay), whereas 2019 emphasizes national identity (american, states, united).

### Step 4: Document Vectorization

In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer

# first, train TfIdf vectorizer on the speech text
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(sou_21cen["Text"])   # (docs × features)

# second, obtain feature vectors 
feature_names = tfidf.get_feature_names_out()
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(),
                        columns=feature_names,
                        index=sou_21cen["Year"])

In [37]:
# print feature vectors
tfidf_df.head()

,000,10,100,100th,102,108,10th,11,110,110th,...,youngstown,youth,zarqawi,zealand,zeitchik,zelenskiy,zero,zimbabwe,zone,zones
Year,,,,,,,,,,,,,,,,,,,,,
2024.0,0.055516,0.018512,0.021694,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.032689,0.0,0.000000,0.000000
2023.0,0.076510,0.044646,0.005232,0.000000,0.0,0.0,0.0,0.017789,0.0,0.0,...,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.015768,0.0,0.000000,0.000000
2022.0,0.044877,0.005237,0.006138,0.000000,0.0,0.0,0.0,0.013912,0.0,0.0,...,0.0,0.000000,0.0,0.015998,0.0,0.031997,0.009249,0.0,0.000000,0.000000
2021.0,0.044969,0.026240,0.067655,0.000000,0.0,0.0,0.0,0.020911,0.0,0.0,...,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.009267,0.0,0.032062,0.000000
2020.0,0.080577,0.023509,0.027552,0.015911,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.012421,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.047734
